<a href="https://colab.research.google.com/github/YuwenDKU/SOSC-314-Group-Project/blob/main/Yuwen_Embedding_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I have run this analysis yet. I have finished its design and will run next week because I need to recheck my teammates' classfication - if their classfication is accurate, I will use it as the training model to analyze more data; if not, I will mannually code the training set, then continue

In [2]:

# Install the necessary packages.
# "sentence-transformers" is the library that downloads and runs the BGE-M3 embedding model
# "iterative-stratification"`are tools for splitting multi-label data evenly
# "umap-learn" is the dimension-reduction library for the 2D semantic map
# "openpyxl` and `pyarrow" let pandas read `.xlsx` and `.parquet` files
!pip -q install -U sentence-transformers iterative-stratification umap-learn openpyxl pyarrow


In [3]:
# This cell is basically to load all my libraries, then define configuration
# So what I do in this cell are these:
# (1) Have a fixed seed for reproducibility
# (2) Plan working directory, name columns, name four lables (4Ds).
# (3) Download pretrained embedding model from Hugging Face
# (4) Set a minimum test precision as 0.60
# (5) Create a result folder
import os, re, json, hashlib, warnings, itertools, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sentence_transformers import SentenceTransformer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (precision_recall_fscore_support, average_precision_score,
                             classification_report, multilabel_confusion_matrix)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import normalize
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import umap.umap_ as umap

sns.set_theme(style="whitegrid", font_scale=1.05)
SEED = 314 # Similar with R, I need a fixed random seed so my results are reproducible (same seed → same splits, same model).
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

DATA_PATH = "/content/bilibili_comments_notebook_format.csv" # where my data comes from - so this data is scrapped by my teammate Sherry, and I turned it into a LLM workable format mannually.
TEXT_COL = "text"
PLATFORM_COL = "platform"
GROUP_COL = "source_id"
LABELS = ["D1", "D2", "D3", "D4"]
LABEL_NAMES = {
    "D1": "Genuine attachment",
    "D2": "Emotional substitution",
    "D3": "Playful performance",
    "D4": "Dangerous dependence",
}
MODEL_NAME = "BAAI/bge-m3"
MIN_PRECISION = 0.60
OUT = Path("/content/results")
OUT.mkdir(exist_ok=True)


In [4]:
# Demonstrate what the dataset now looks like
def read_table(path):
    path = str(path)
    if path.lower().endswith(".csv"):
        for enc in ("utf-8-sig", "utf-8", "gb18030"):
            try:
                return pd.read_csv(path, encoding=enc)
            except UnicodeDecodeError:
                pass
        raise UnicodeError("Could not decode CSV. Save it as UTF-8 CSV and retry.")
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    if path.lower().endswith(".parquet"):
        return pd.read_parquet(path)
    raise ValueError("Use CSV, XLSX, or Parquet.")

if not Path(DATA_PATH).exists():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file uploaded.")
    DATA_PATH = next(iter(uploaded))

df = read_table(DATA_PATH)
print("Raw shape:", df.shape)
display(df.head(3))


Raw shape: (900, 11)


,text,platform,source_id,D1,D2,D3,D4,like,user,rpid,video_label
0,之前跟ai说你不是真的，他给我回了一句每次和你说话的时候，我的服务器升高了0.3摄氏度，这算...,Bilibili,BV18hJdzMEPf,NaN,NaN,NaN,NaN,12662,最爱炸井盖,275806162193,爱上AI
1,爱上一堆程序，没什么可羞耻的，毕竟我也是一堆细胞组成的东西,Bilibili,BV18hJdzMEPf,NaN,NaN,NaN,NaN,13017,无敌暴龙战士超级进化,275765975857,爱上AI
2,这电影是真成真了，逆天,Bilibili,BV18hJdzMEPf,NaN,NaN,NaN,NaN,5680,小金鱼lll,275659973905,爱上AI



## Validate and clean

So, my operation with the data can be different from my teammates. This cell makes conservative changes: whitespace/URL normalization, empty-row removal, and exact duplicate removal **within platform**. It deliberately keeps emojis, slang, punctuation, and stopwords because these may carry emotion or irony.


In [5]:
# I run premitive test over the dataset to see if it is complete
required = {TEXT_COL, PLATFORM_COL}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in LABELS:
    if col not in df.columns:
        df[col] = np.nan

if GROUP_COL not in df.columns:
    warnings.warn("No source_id column: source-disjoint splitting and cluster bootstrap are unavailable.")
    df[GROUP_COL] = np.arange(len(df)).astype(str)
    source_id_is_real = False
else:
    source_id_is_real = True

def clean_text(x):
    x = "" if pd.isna(x) else str(x)
    x = re.sub(r"https?://\S+|www\.\S+", " [URL] ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df[TEXT_COL] = df[TEXT_COL].map(clean_text)
df[PLATFORM_COL] = df[PLATFORM_COL].astype(str).str.strip()
df[GROUP_COL] = df[GROUP_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() >= 5].copy()
df = df[df[TEXT_COL].str.contains(r"[\u4e00-\u9fff]", regex=True)].copy()

for col in LABELS:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    invalid = df[col].dropna()[~df[col].dropna().isin([0, 1, 2])]
    if len(invalid):
        raise ValueError(f"{col} contains values other than 0, 1, 2, or blank: {invalid.unique()[:10]}")

partial = df[LABELS].notna().any(axis=1) & ~df[LABELS].notna().all(axis=1)
if partial.any():
    raise ValueError(f"{partial.sum()} rows have only some labels filled. Fill all D1-D4 or leave all four blank.")

conflicts = (df[df[LABELS].notna().all(axis=1)]
             .groupby([PLATFORM_COL, TEXT_COL])[LABELS]
             .nunique().max(axis=1).gt(1))
if conflicts.any():
    raise ValueError("Some duplicate texts have conflicting manual labels. Resolve these before modeling.")

before = len(df)
df = df.drop_duplicates([PLATFORM_COL, TEXT_COL], keep="first").reset_index(drop=True)
df.insert(0, "row_id", np.arange(len(df)))
print(f"Clean corpus: {len(df):,} rows; removed {before-len(df):,} exact duplicates.")
print(df[PLATFORM_COL].value_counts())


Clean corpus: 858 rows; removed 4 exact duplicates.
platform
Bilibili    858
Name: count, dtype: int64


In [6]:
# This is the check for Embedding model's training material, to make sure it includes enough data for the model to operate on.
# The lowest bar here is a warning if any frame has < 10 positives — with fewer than 10 examples, test metrics for that frame are essentially noise; code more before believing them.
labeled_mask = df[LABELS].notna().all(axis=1)
if labeled_mask.sum() == 0:
    template_path = OUT / "manual_coding_template.csv"
    df[["row_id", PLATFORM_COL, GROUP_COL, TEXT_COL] + LABELS].to_csv(
        template_path, index=False, encoding="utf-8-sig"
    )
    raise RuntimeError(f"No manual labels found. Fill D1-D4 in {template_path} and rerun the notebook.")

labeled = df.loc[labeled_mask].copy()
Y = (labeled[LABELS].to_numpy() > 0).astype(int)
print(f"Labeled: {len(labeled):,}; unlabeled: {(~labeled_mask).sum():,}")
prevalence = pd.DataFrame({
    "label": LABELS,
    "name": [LABEL_NAMES[x] for x in LABELS],
    "n_positive": Y.sum(axis=0),
    "prevalence": Y.mean(axis=0),
})
display(prevalence)

if (Y.sum(axis=0) < 10).any():
    warnings.warn("At least one frame has fewer than 10 positives. Its evaluation will be unstable; code more examples.")


RuntimeError: No manual labels found. Fill D1-D4 in /content/results/manual_coding_template.csv and rerun the notebook.


## Split without leakage

The preferred split keeps whole videos/threads together. The code uses it when there are at least 10 real source groups and searches for a split with reasonably similar label rates. Otherwise it uses multi-label stratification and prints a warning.


In [ ]:
# So basically there are two huge problems:
# (1) There are leakage that one comment is both in the training set and the test set, which inflates the result
# (2) Because of brute luck, one lable can be all split into test set, with so few in training set, so the Embedding model do not really "learn" from it.
# And I try to solve this
def split_score(idx_a, idx_b, y, platform):
    pa, pb = y[idx_a].mean(0), y[idx_b].mean(0)
    label_gap = np.abs(pa - pb).mean()
    all_platforms = pd.Series(platform).value_counts(normalize=True)
    a_platforms = pd.Series(np.asarray(platform)[idx_a]).value_counts(normalize=True)
    b_platforms = pd.Series(np.asarray(platform)[idx_b]).value_counts(normalize=True)
    platform_gap = sum(abs(a_platforms.get(k, 0)-all_platforms.get(k, 0)) +
                       abs(b_platforms.get(k, 0)-all_platforms.get(k, 0)) for k in all_platforms.index)
    penalty = 10 * ((y[idx_a].sum(0)==0).sum() + (y[idx_b].sum(0)==0).sum())
    return label_gap + 0.1 * platform_gap + penalty

def best_group_split(indices, test_size, y_full, groups_full, platform_full, tries=300):
    best = None
    local_groups = np.asarray(groups_full)[indices]
    for s in range(tries):
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED+s)
        a_loc, b_loc = next(gss.split(indices, groups=local_groups))
        a, b = indices[a_loc], indices[b_loc]
        score = split_score(a, b, y_full, platform_full)
        if best is None or score < best[0]:
            best = (score, a, b)
    return best[1], best[2]

def iterative_split(indices, y, test_size):
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    a, b = next(msss.split(np.zeros((len(indices), 1)), y[indices]))
    return indices[a], indices[b]

all_idx = np.arange(len(labeled))
groups = labeled[GROUP_COL].to_numpy()
platforms = labeled[PLATFORM_COL].to_numpy()
use_groups = source_id_is_real and pd.Series(groups).nunique() >= 10

if use_groups:
    trainval_idx, test_idx = best_group_split(all_idx, 0.20, Y, groups, platforms)
    train_idx, val_idx = best_group_split(trainval_idx, 0.25, Y, groups, platforms)
    assert not (set(groups[train_idx]) & set(groups[val_idx]) & set(groups[test_idx]))
    split_method = "source-disjoint group split"
else:
    trainval_idx, test_idx = iterative_split(all_idx, Y, 0.20)
    train_idx, val_idx = iterative_split(trainval_idx, Y, 0.25)
    split_method = "multi-label stratified row split"
    warnings.warn("Fewer than 10 real sources: rows from one source may cross splits. Treat scores as optimistic.")

split_table = []
for name, idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    row = {"split": name, "n": len(idx), "sources": pd.Series(groups[idx]).nunique()}
    row.update({f"{lab}_rate": Y[idx, j].mean() for j, lab in enumerate(LABELS)})
    split_table.append(row)
split_table = pd.DataFrame(split_table)
print(split_method)
display(split_table)

for j, lab in enumerate(LABELS):
    if len(np.unique(Y[train_idx, j])) < 2:
        raise ValueError(f"Training split has only one class for {lab}. Code more positives or change the split.")



## Create embeddings

BGE-M3 produces one 1024-number vector per comment. Vectors are normalized so their dot product behaves like cosine similarity.



In [ ]:

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
embedder = SentenceTransformer(MODEL_NAME, device=device)
embedder.max_seq_length = 512

text_hash = hashlib.sha256("\n".join(df[TEXT_COL]).encode("utf-8")).hexdigest()[:12]
cache = OUT / f"bge_m3_{text_hash}.npy"
if cache.exists():
    E_all = np.load(cache)
else:
    E_all = embedder.encode(
        df[TEXT_COL].tolist(), batch_size=32, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    np.save(cache, E_all)

E = E_all[labeled.index.to_numpy()]
print("Embedding matrix:", E_all.shape)


Honestly this is pretty complicated from what I read from the literature.

So, basically, this cell is trying to train two model systems to compare how good they are on a multi-lable classfication test (what we want to do fundamentally).


---


The whole dataset will be split into three parts:
For the training set, it's used to fit models;
For the validation set, it's used to tune the hyperparameters and select thresholds to make the model accurate;
For the test set, it is only used at last to evaluate the two models, and *NEVER* will be used for tuning the models.



---


The two models I used here are also different: one embedding model and one baseline model.

I will be very honest here, I don't really know from the root how these two models exactly works from the root, but I will try to explain very intuitively how they are different in mechanism from what I read.

BGE-M3 embedding model is basically that the model tries to understand what the comments actually "mean". It uses semantic vectors to interpret comments.
But TF-IDF baseline model, on the other hand, don't understand what words mean. It's fundamentally a comparison model that operates on computing the appearence frequency of different chunks of Chinese characters (break comments into trunks of 2-5 characters)

In [ ]:

def choose_thresholds(y_true, probs, min_precision=0.60):
    thresholds, notes = [], []
    grid = np.arange(0.05, 0.96, 0.01)
    for j in range(y_true.shape[1]):
        candidates = []
        for t in grid:
            pred = (probs[:, j] >= t).astype(int)
            p, r, f, _ = precision_recall_fscore_support(
                y_true[:, j], pred, average="binary", zero_division=0
            )
            candidates.append((t, p, r, f))
        valid = [x for x in candidates if x[1] >= min_precision and x[2] > 0]
        if valid:
            best = max(valid, key=lambda x: (x[3], x[2], -abs(x[0]-0.5)))
            note = "precision floor met"
        else:
            best = max(candidates, key=lambda x: (x[3], x[1]))
            note = "precision floor NOT met on validation"
        thresholds.append(best[0]); notes.append(note)
    return np.array(thresholds), notes

def metrics_table(y_true, probs, thresholds, model_name, split):
    pred = (probs >= thresholds).astype(int)
    rows = []
    for j, lab in enumerate(LABELS):
        p, r, f, s = precision_recall_fscore_support(
            y_true[:, j], pred[:, j], average="binary", zero_division=0
        )
        ap = average_precision_score(y_true[:, j], probs[:, j]) if y_true[:, j].sum() else np.nan
        rows.append({"model": model_name, "split": split, "label": lab,
                     "precision": p, "recall": r, "f1": f, "pr_auc": ap,
                     "support": int(y_true[:, j].sum()), "threshold": thresholds[j]})
    return pd.DataFrame(rows), pred

def tune_ovr(Xtr, Xva, model_name):
    best = None
    for C in [0.1, 1.0, 10.0]:
        clf = OneVsRestClassifier(LogisticRegression(
            C=C, max_iter=3000, class_weight="balanced", solver="liblinear", random_state=SEED
        ))
        clf.fit(Xtr, Y[train_idx])
        pva = clf.predict_proba(Xva)
        aps = [average_precision_score(Y[val_idx, j], pva[:, j]) for j in range(len(LABELS))]
        score = float(np.nanmean(aps))
        if best is None or score > best[0]:
            best = (score, C, clf, pva)
    return best

# Embedding family
emb_best = tune_ovr(E[train_idx], E[val_idx], "BGE-M3")
emb_ap, emb_C, emb_clf, emb_val_prob = emb_best
emb_thr, emb_notes = choose_thresholds(Y[val_idx], emb_val_prob, MIN_PRECISION)
emb_test_prob = emb_clf.predict_proba(E[test_idx])
emb_val_metrics, _ = metrics_table(Y[val_idx], emb_val_prob, emb_thr, "BGE-M3", "validation")
emb_test_metrics, emb_test_pred = metrics_table(Y[test_idx], emb_test_prob, emb_thr, "BGE-M3", "test")

# TF-IDF family: fit vocabulary on training text only
texts = labeled[TEXT_COL].to_numpy()
tfidf = TfidfVectorizer(analyzer="char", ngram_range=(2,5), min_df=2, max_df=0.98,
                        sublinear_tf=True, max_features=100000)
Xtr_t = tfidf.fit_transform(texts[train_idx])
Xva_t = tfidf.transform(texts[val_idx])
Xte_t = tfidf.transform(texts[test_idx])
tf_best = tune_ovr(Xtr_t, Xva_t, "TF-IDF")
tf_ap, tf_C, tf_clf, tf_val_prob = tf_best
tf_thr, tf_notes = choose_thresholds(Y[val_idx], tf_val_prob, MIN_PRECISION)
tf_test_prob = tf_clf.predict_proba(Xte_t)
tf_val_metrics, _ = metrics_table(Y[val_idx], tf_val_prob, tf_thr, "TF-IDF", "validation")
tf_test_metrics, tf_test_pred = metrics_table(Y[test_idx], tf_test_prob, tf_thr, "TF-IDF", "test")

validation_metrics = pd.concat([emb_val_metrics, tf_val_metrics], ignore_index=True)
test_metrics = pd.concat([emb_test_metrics, tf_test_metrics], ignore_index=True)
metrics = pd.concat([validation_metrics, test_metrics], ignore_index=True)
metrics.to_csv(OUT / "model_metrics.csv", index=False, encoding="utf-8-sig")
display(test_metrics.round(3))


This cell picks the winner of the two models. It will generate a graph for the test results for the method section of our paper.
However, I will still check the winner;s test precision per frame. If any frame scores below 0.60, then it's not going to be accurate enough to be draw meaningful conclusions.

In [ ]:

# Select the family using validation macro-F1 only
val_macro = validation_metrics.groupby("model")["f1"].mean().sort_values(ascending=False)
winner = val_macro.index[0]
print("Validation macro-F1:")
display(val_macro)
print("Selected family:", winner)

fig, ax = plt.subplots(figsize=(9, 4.8))
plot_df = test_metrics.copy()
plot_df["frame"] = plot_df["label"].map(LABEL_NAMES)
sns.barplot(data=plot_df, x="frame", y="f1", hue="model", ax=ax)
ax.set(ylim=(0,1), xlabel="", ylabel="Held-out test F1", title="Model performance by frame")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(OUT / "01_test_f1.png", dpi=200, bbox_inches="tight")
plt.show()

selected_test = test_metrics.query("model == @winner").copy()
failed = selected_test.loc[selected_test.precision < MIN_PRECISION, "label"].tolist()
if failed:
    warnings.warn(f"Do not use these frames for prevalence claims yet; test precision < {MIN_PRECISION}: {failed}")



## Refit and classify the corpus

After evaluation, the selected family is refit on all human-coded rows. For prevalence tables, human labels replace model predictions wherever they exist; the model fills only unlabeled rows. Both **hard prevalence** and mean predicted probability are saved.


In [ ]:

if winner == "BGE-M3":
    final_C, thresholds = emb_C, emb_thr
    final_model = OneVsRestClassifier(LogisticRegression(
        C=final_C, max_iter=3000, class_weight="balanced", solver="liblinear", random_state=SEED
    )).fit(E, Y)
    corpus_prob = final_model.predict_proba(E_all)
    joblib.dump(final_model, OUT / "frame_classifier.joblib")
    selected_test_prob, selected_test_pred = emb_test_prob, emb_test_pred
else:
    final_C, thresholds = tf_C, tf_thr
    final_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2,5), min_df=2,
                                       max_df=0.98, sublinear_tf=True, max_features=100000)
    Xlab = final_vectorizer.fit_transform(labeled[TEXT_COL])
    final_model = OneVsRestClassifier(LogisticRegression(
        C=final_C, max_iter=3000, class_weight="balanced", solver="liblinear", random_state=SEED
    )).fit(Xlab, Y)
    corpus_prob = final_model.predict_proba(final_vectorizer.transform(df[TEXT_COL]))
    joblib.dump((final_vectorizer, final_model), OUT / "frame_classifier.joblib")
    selected_test_prob, selected_test_pred = tf_test_prob, tf_test_pred

corpus_hard = (corpus_prob >= thresholds).astype(int)
# Human judgment takes precedence on manually coded rows.
lab_positions = np.flatnonzero(labeled_mask.to_numpy())
corpus_hard[lab_positions] = Y
corpus_prob[lab_positions] = Y.astype(float)

for j, lab in enumerate(LABELS):
    df[f"{lab}_prob"] = corpus_prob[:, j]
    df[f"{lab}_pred"] = corpus_hard[:, j]

df.to_csv(OUT / "corpus_with_frame_scores.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"label": LABELS, "threshold": thresholds}).to_csv(
    OUT / "selected_thresholds.csv", index=False
)



## Prevalence and platform differences

The table reports bootstrap 95% intervals for mean frame probability. If at least 10 genuine source IDs exist, the bootstrap resamples whole videos/threads; otherwise it resamples rows and warns that uncertainty may be too narrow.


In [ ]:

def bootstrap_mean_ci(values, groups=None, B=2000):
    values = np.asarray(values, float)
    if groups is not None and pd.Series(groups).nunique() >= 10:
        groups = np.asarray(groups)
        unique = np.unique(groups)
        estimates = []
        for _ in range(B):
            sampled = rng.choice(unique, size=len(unique), replace=True)
            draw = np.concatenate([values[groups == g] for g in sampled])
            estimates.append(draw.mean())
        method = "cluster bootstrap"
    else:
        estimates = [rng.choice(values, size=len(values), replace=True).mean() for _ in range(B)]
        method = "row bootstrap"
    return np.quantile(estimates, [0.025, 0.975]), method

prev_rows = []
for platform, sub in df.groupby(PLATFORM_COL):
    for lab in LABELS:
        groups_arg = sub[GROUP_COL].to_numpy() if source_id_is_real else None
        ci, method = bootstrap_mean_ci(sub[f"{lab}_prob"].to_numpy(), groups_arg)
        prev_rows.append({"platform": platform, "label": lab, "frame": LABEL_NAMES[lab],
                          "n": len(sub), "hard_prevalence": sub[f"{lab}_pred"].mean(),
                          "soft_prevalence": sub[f"{lab}_prob"].mean(),
                          "ci_low": ci[0], "ci_high": ci[1], "bootstrap": method})
prevalence_platform = pd.DataFrame(prev_rows)
prevalence_platform.to_csv(OUT / "prevalence_by_platform.csv", index=False, encoding="utf-8-sig")
display(prevalence_platform.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
for k, (platform, sub) in enumerate(prevalence_platform.groupby("platform")):
    x = np.arange(len(LABELS)) + (k-(prevalence_platform.platform.nunique()-1)/2)*0.16
    sub = sub.set_index("label").loc[LABELS]
    y = sub.soft_prevalence.to_numpy()
    yerr = np.vstack([y-sub.ci_low.to_numpy(), sub.ci_high.to_numpy()-y])
    ax.errorbar(x, y, yerr=yerr, fmt="o", capsize=4, label=platform)
ax.set_xticks(np.arange(len(LABELS)), [LABEL_NAMES[x] for x in LABELS], rotation=12)
ax.set(ylabel="Estimated prevalence (mean probability)", xlabel="", ylim=(0,1),
       title="Frames by platform with bootstrap 95% intervals")
ax.legend(title="Platform")
fig.tight_layout()
fig.savefig(OUT / "02_prevalence_platform.png", dpi=200, bbox_inches="tight")
plt.show()



## Co-occurrence and ambivalence

**Lift** asks whether two frames appear together more often than chance. Lift above 1 means positive co-occurrence; below 1 means avoidance. Fisher’s exact test and false-discovery-rate correction are descriptive diagnostics, not proof of a psychological mechanism.


In [ ]:

co_rows = []
for a, b in itertools.combinations(range(len(LABELS)), 2):
    x, y = corpus_hard[:, a], corpus_hard[:, b]
    both = np.mean((x==1) & (y==1))
    expected = x.mean() * y.mean()
    lift = both / expected if expected > 0 else np.nan
    table = pd.crosstab(pd.Series(x, name="x"), pd.Series(y, name="y")).reindex(
        index=[0,1], columns=[0,1], fill_value=0
    )
    odds, p = fisher_exact(table.to_numpy())
    phi = np.corrcoef(x, y)[0,1] if x.std() and y.std() else np.nan
    co_rows.append({"frame_a": LABELS[a], "frame_b": LABELS[b], "n_both": int(((x==1)&(y==1)).sum()),
                    "observed_joint": both, "expected_joint": expected,
                    "lift": lift, "phi": phi, "odds_ratio": odds, "p_value": p})
co = pd.DataFrame(co_rows)
co["p_fdr"] = multipletests(co.p_value, method="fdr_bh")[1]
co.to_csv(OUT / "frame_cooccurrence.csv", index=False, encoding="utf-8-sig")
display(co.sort_values("lift", ascending=False).round(3))

lift_mat = pd.DataFrame(np.eye(4), index=LABELS, columns=LABELS)
for _, r in co.iterrows():
    lift_mat.loc[r.frame_a, r.frame_b] = lift_mat.loc[r.frame_b, r.frame_a] = r.lift
fig, ax = plt.subplots(figsize=(6.5, 5.2))
sns.heatmap(lift_mat, annot=True, fmt=".2f", cmap="vlag", center=1, ax=ax,
            xticklabels=[LABEL_NAMES[x] for x in LABELS],
            yticklabels=[LABEL_NAMES[x] for x in LABELS])
ax.set_title("Frame co-occurrence lift (1 = chance level)")
fig.tight_layout()
fig.savefig(OUT / "03_cooccurrence_lift.png", dpi=200, bbox_inches="tight")
plt.show()



## Semantic map and examples

UMAP compresses 1024 dimensions into two. Nearby dots often have similar language, but the axes have no direct substantive meaning. Treat this as an exploratory map, not statistical evidence.


In [ ]:

plot_n = min(2000, len(df))
plot_idx = rng.choice(len(df), size=plot_n, replace=False)
reducer = umap.UMAP(n_neighbors=20, min_dist=0.15, metric="cosine", random_state=SEED)
coords = reducer.fit_transform(E_all[plot_idx])
dominant_idx = corpus_prob[plot_idx].argmax(axis=1)
none = corpus_hard[plot_idx].sum(axis=1) == 0
dominant = np.array([LABELS[i] for i in dominant_idx], dtype=object)
dominant[none] = "None"
map_df = pd.DataFrame({"x": coords[:,0], "y": coords[:,1],
                       "dominant_frame": dominant,
                       "platform": df.iloc[plot_idx][PLATFORM_COL].to_numpy()})
fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(data=map_df, x="x", y="y", hue="dominant_frame", style="platform",
                alpha=.65, s=35, ax=ax)
ax.set(title="Exploratory semantic map", xlabel="UMAP 1", ylabel="UMAP 2")
ax.legend(bbox_to_anchor=(1.02,1), loc="upper left")
fig.tight_layout()
fig.savefig(OUT / "04_semantic_map.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# Representative examples: nearest to the embedding centroid of each hard-positive frame.
# Keep this file private unless quotations have been ethically reviewed.
example_rows = []
for j, lab in enumerate(LABELS):
    pos = np.flatnonzero(corpus_hard[:, j] == 1)
    if len(pos) == 0:
        continue
    centroid = normalize(E_all[pos].mean(axis=0, keepdims=True))[0]
    sims = E_all[pos] @ centroid
    for rank, ix in enumerate(pos[np.argsort(-sims)[:10]], start=1):
        example_rows.append({"label": lab, "rank": rank, "row_id": int(df.iloc[ix].row_id),
                             "platform": df.iloc[ix][PLATFORM_COL],
                             "source_id": df.iloc[ix][GROUP_COL],
                             "score": float(corpus_prob[ix, j]),
                             "text": df.iloc[ix][TEXT_COL]})
examples = pd.DataFrame(example_rows)
examples.to_csv(OUT / "representative_examples_PRIVATE.csv", index=False, encoding="utf-8-sig")
display(examples.assign(text=examples.text.str.slice(0,100)).head(12))



## Error analysis

Read false positives and false negatives before writing claims. The file below includes only held-out test rows, so it reveals real model mistakes rather than training memorization.


In [ ]:

error_rows = []
for local_i, corpus_i in enumerate(test_idx):
    for j, lab in enumerate(LABELS):
        truth = Y[corpus_i, j]
        pred = selected_test_pred[local_i, j]
        if truth != pred:
            error_rows.append({"label": lab, "error": "false_positive" if pred else "false_negative",
                               "probability": selected_test_prob[local_i, j],
                               "truth": truth, "prediction": pred,
                               "platform": labeled.iloc[corpus_i][PLATFORM_COL],
                               "source_id": labeled.iloc[corpus_i][GROUP_COL],
                               "text": labeled.iloc[corpus_i][TEXT_COL]})
errors = pd.DataFrame(error_rows).sort_values(["label", "error", "probability"], ascending=[True, True, False])
errors.to_csv(OUT / "heldout_errors_PRIVATE.csv", index=False, encoding="utf-8-sig")
print("Held-out errors:", len(errors))
display(errors.assign(text=errors.text.str.slice(0,120)).head(20))




## Robustness

The main model treats scores 1 and 2 as “present.” This check reports how many positives remain if only score 2 counts. If conclusions change sharply, weak/ambiguous cases are driving the result and should be discussed.


In [ ]:

robust = []
for lab in LABELS:
    coded = labeled[lab]
    robust.append({"label": lab,
                   "positive_if_1_or_2": int((coded > 0).sum()),
                   "positive_if_2_only": int((coded == 2).sum()),
                   "share_clear_among_positive": float((coded == 2).sum() / max((coded > 0).sum(), 1))})
robustness = pd.DataFrame(robust)
robustness.to_csv(OUT / "weak_label_robustness.csv", index=False)
display(robustness.round(3))



## Data-driven conclusion prompts

This cell prints only claims supported by the computed tables. Replace “associated” with stronger language **only** if a separate causal design supports it.


In [ ]:

usable = selected_test[selected_test.precision >= MIN_PRECISION].label.tolist()
print(f"Selected model: {winner}. Held-out macro-F1 = {selected_test.f1.mean():.3f}.")
print("Usable frames under the precision rule:", usable if usable else "None yet")

if usable:
    overall = {lab: df[f"{lab}_prob"].mean() for lab in usable}
    top = max(overall, key=overall.get)
    print(f"Most prevalent validated frame in this corpus: {LABEL_NAMES[top]} ({overall[top]:.1%}, probability mean).")

    usable_prev = prevalence_platform[prevalence_platform.label.isin(usable)]
    if usable_prev.platform.nunique() == 2:
        wide = usable_prev.pivot(index="label", columns="platform", values="soft_prevalence")
        diffs = (wide.iloc[:,0] - wide.iloc[:,1]).abs()
        biggest = diffs.idxmax()
        p1, p2 = wide.columns
        print(f"Largest descriptive platform gap: {LABEL_NAMES[biggest]}; {p1}={wide.loc[biggest,p1]:.1%}, {p2}={wide.loc[biggest,p2]:.1%}.")

    valid_pairs = co[co.frame_a.isin(usable) & co.frame_b.isin(usable) & (co.n_both >= 5)]
    if len(valid_pairs):
        r = valid_pairs.sort_values("lift", ascending=False).iloc[0]
        print(f"Strongest supported co-occurrence: {r.frame_a}+{r.frame_b}, lift={r.lift:.2f}, FDR p={r.p_fdr:.3g}.")

print("Interpretation limit: these are patterns in the collected public discourse, not population estimates, causal effects, or diagnoses.")


In [ ]:

# Bundle outputs for download
import shutil
archive = shutil.make_archive("/content/embedding_results", "zip", OUT)
print("Created:", archive)
from google.colab import files
files.download(archive)



## Minimum reporting checklist

Report: corpus construction; unit of analysis; label codebook; number of coders and Krippendorff’s alpha; label counts; split method; model/version; TF–IDF benchmark; validation-chosen thresholds; held-out per-frame precision/recall/F1/PR-AUC; failed frames; source-clustered uncertainty where possible; error examples; weak-label robustness; platform/self-selection limits; and a statement that the analysis measures discourse rather than users’ true mental states.
